# Netflix Top 10 Weekly Dataset — EDA

기간: 2021-07-04 ~ 2022-01-16 (29주) · 컬럼 정의는 `references/data_dictionary.md` 참고

**순서**: 데이터 구조 → 1.결측치 → 2.이상치/범위 → 3.분포/불균형 → 4.상관관계

그림은 `reports/figures/`에 저장됩니다.

In [ ]:
# Netflix Top 10 Weekly - EDA
# 데이터: data/raw (원본, 수정 금지) | 그림 저장: reports/figures

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"

# countries는 cp1252 인코딩 (references/data_dictionary.md 참고)
world = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", parse_dates=["week"])
country = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", encoding="cp1252", parse_dates=["week"])
print(world.shape, country.shape)
world.head()

In [ ]:
# 데이터 구조 요약
print("world  :", world.shape, "| 주:", world.week.nunique(), "| 카테고리:", world.category.nunique(), "| 작품:", world.show_title.nunique())
print("country:", country.shape, "| 국가:", country.country_name.nunique(), "| 카테고리:", country.category.nunique(), "| 작품:", country.show_title.nunique())
print()
print("world 행 구성:", world.category.value_counts().to_dict())
print("country 행 구성:", country.category.value_counts().to_dict())

## 1. 결측치의 비율과 위치

In [ ]:
# 1. 결측치 비율과 위치
print(world.isna().sum()[lambda s: s > 0], "\n")
print(country.isna().sum()[lambda s: s > 0], "\n")

# 위치: season_title 결측은 영화(시즌 없음)에서 발생한 구조적 결측
w_t = world[world.category.str.startswith("TV")]
c_t = country[country.category == "TV"]
print("world  TV 중 season_title 결측:", w_t.season_title.isna().sum(), "행 ->", sorted(w_t[w_t.season_title.isna()].show_title.unique()))
print("country TV 중 season_title 결측:", c_t.season_title.isna().sum(), "행 ->", c_t[c_t.season_title.isna()].show_title.nunique(), "개 작품")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
vals = [100.0, 100.0, w_t.season_title.isna().mean() * 100, 0.0]
axes[0].bar(["Films(EN)", "Films(non-EN)", "TV(EN)", "TV(non-EN)"], vals, color="#e50914")
axes[0].set_title("[world] season_title 결측 비율(%) - 카테고리별")
for i, v in enumerate(vals):
    axes[0].text(i, v + 2, f"{v:.1f}%", ha="center", fontsize=9)
vals2 = [100.0, c_t.season_title.isna().mean() * 100]
axes[1].bar(["Films", "TV"], vals2, color="#e50914")
axes[1].set_title("[country] season_title 결측 비율(%) - 카테고리별")
for i, v in enumerate(vals2):
    axes[1].text(i, v + 2, f"{v:.1f}%", ha="center", fontsize=9)
caption = ("실제 결측이 아님: 영화는 시즌(시리즈)이 없어 원본 CSV의 'N/A' 표시가 NaN으로 읽힌 구조적 결측 -> "
           "별도 처리 없이 유지해도 무방")
fig.text(0.5, 0.005, caption, ha="center", va="bottom", fontsize=10, color="#333333")
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.savefig(FIG / "01_missing.png", dpi=130)
plt.show()

## 2. 이상치와 값의 범위

In [ ]:
# 2. 이상치와 값의 범위
h = world.weekly_hours_viewed
q1, q3 = h.quantile(0.25), h.quantile(0.75)
hi = q3 + 1.5 * (q3 - q1)
out = world[h > hi]
print(f"범위: {h.min():,} ~ {h.max():,} 시간 | IQR 상한: {hi:,.0f} | 이상치: {len(out)}행 ({len(out)/len(world)*100:.1f}%)")
print(out[["week", "category", "show_title", "weekly_hours_viewed", "weekly_rank"]].nlargest(5, "weekly_hours_viewed").to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].hist(h, bins=40, color="#e50914", edgecolor="white")
axes[0].axvline(hi, color="black", ls="--", lw=1)
axes[0].text(hi * 1.05, axes[0].get_ylim()[1] * 0.9, "IQR 상한\n약 4,000만", fontsize=9)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: "0" if v == 0 else f"{v/1e8:g}억"))
axes[0].set_title("[world] weekly_hours_viewed 분포")
axes[0].set_xlabel("주간 시청시간 (억 시간)")
axes[0].set_ylabel("행 수")
sns.boxplot(x=h, ax=axes[1], color="#e50914")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: "0" if v == 0 else f"{v/1e8:g}억"))
axes[1].set_xlabel("주간 시청시간 (억 시간)")
axes[1].set_title("[world] weekly_hours_viewed boxplot (IQR 이상치 = 점)")
plt.tight_layout()
plt.savefig(FIG / "02_outliers.png", dpi=130)
plt.show()

In [ ]:
# 2b. 대안 시각화: 히트맵(카테고리 x 순위) + 밀도 곡선
cats = ["Films (English)", "Films (Non-English)", "TV (English)", "TV (Non-English)"]
pv = (world.groupby(["category", "weekly_rank"]).weekly_hours_viewed.median()
      .unstack("weekly_rank").loc[cats])
fmt = lambda v: f"{v/1e8:.1f}억" if v >= 1e8 else f"{v/1e4:,.0f}만"

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
sns.heatmap(pv, ax=axes[0], cmap="Reds", annot=pv.map(fmt).values, fmt="",
            norm=matplotlib.colors.LogNorm(vmin=pv.values.min(), vmax=pv.values.max()),
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "중앙값 주간 시청시간 (시간, 로그 색)"})
axes[0].set_title("카테고리 × 순위별 주간 시청시간 (중앙값, 히트맵)")
axes[0].set_xlabel("주간 순위")
axes[0].tick_params(axis="y", rotation=0)

med = h.median()
sns.kdeplot(x=h, log_scale=True, fill=True, color="#e50914", ax=axes[1])
for v, ls, lab in [(q1, ":", f"Q1 {fmt(q1)}"), (med, "-", f"중앙값 {fmt(med)}"),
                   (q3, ":", f"Q3 {fmt(q3)}"), (hi, "--", f"IQR 상한 {fmt(hi)}")]:
    axes[1].axvline(v, color="#333333", ls=ls, lw=1.2, label=lab)
axes[1].set_xlim(4e5, 1.2e9)
axes[1].xaxis.set_major_locator(mticker.FixedLocator([1e6, 1e7, 1e8]))
axes[1].xaxis.set_major_formatter(mticker.FixedFormatter(["100만", "1,000만", "1억"]))
axes[1].xaxis.set_minor_formatter(mticker.NullFormatter())
axes[1].set_xlabel("주간 시청시간 (시간, 로그 축)")
axes[1].set_ylabel("밀도")
axes[1].set_title("weekly_hours_viewed 밀도 곡선 (선)")
axes[1].legend(fontsize=8.5)
plt.tight_layout()
plt.savefig(FIG / "02b_hours_alt.png", dpi=130)
plt.show()

## 3. 대상 변수의 분포와 클래스 불균형 (target = `weekly_rank` 가정)

In [ ]:
# 3. 대상 변수의 분포와 클래스 불균형 (target = weekly_rank 가정)
print("weekly_rank 분포 (world):", world.weekly_rank.value_counts().sort_index().to_dict())
print("weekly_rank 분포 (country):", country.weekly_rank.value_counts().sort_index().to_dict())
print("category (world):", world.category.value_counts().to_dict())

freq = world.show_title.value_counts()
print(f"고유 작품 {len(freq)}편 중 1주만 등장: {(freq == 1).sum()}편 ({(freq == 1).mean()*100:.1f}%) | 10주+ 롱런: {(freq >= 10).sum()}편 ({(freq >= 10).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
world.weekly_rank.value_counts().sort_index().plot.bar(ax=axes[0], color="#e50914")
axes[0].set_title("[world] weekly_rank 분포 (균등)")
world.category.value_counts().plot.bar(ax=axes[1], color="#221f1f")
axes[1].set_title("[world] category 분포 (균등)")
axes[1].tick_params(axis="x", rotation=20)
axes[2].hist(freq, bins=20, color="#e50914", edgecolor="white")
axes[2].set_title("[world] 작품별 등장 주 수 (고유 401편)")
axes[2].set_xlabel("등장 주 수")
plt.tight_layout()
plt.savefig(FIG / "03_target.png", dpi=130)
plt.show()

## 4. 변수 사이의 상관관계

In [ ]:
# 4. 변수 사이의 상관관계
num_w = world[["weekly_rank", "weekly_hours_viewed", "cumulative_weeks_in_top_10"]]
num_c = country[["weekly_rank", "cumulative_weeks_in_top_10"]]
print("[world] Pearson:")
print(num_w.corr().round(3), "\n")
print("[country] Pearson:")
print(num_c.corr().round(3), "\n")
r, p = stats.pearsonr(world.weekly_rank, world.weekly_hours_viewed)
print(f"rank ~ hours: r={r:.3f} (p={p:.1e}) -> 순위 숫자가 커질수록(1위->10위) 시청시간 감소 = 순위가 좋을수록 시청시간 많음")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.heatmap(num_w.corr(), annot=True, fmt=".3f", cmap="RdBu_r", vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title("[world] 수치형 상관관계")
sns.heatmap(num_c.corr(), annot=True, fmt=".3f", cmap="RdBu_r", vmin=-1, vmax=1, ax=axes[1], square=True)
axes[1].set_title("[country] 수치형 상관관계")
caption = ("음의 상관(r=-0.418)의 의미: 순위 숫자가 커질수록(1위->10위로 밀릴수록) 시청시간이 줄어듦 -> "
           "실질적으로는 '순위가 좋을수록 시청시간이 많다'는 양의 관계")
fig.text(0.5, 0.005, caption, ha="center", va="bottom", fontsize=10, color="#333333")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(FIG / "04_corr.png", dpi=130)
plt.show()